# 06 — DAB-positive cell counting

`rp.count_cells` counts brown (DAB-positive) cells on IHC slides: an HSV color
gate finds brown pixels and a per-patch Otsu threshold separates cells from
lighter staining. Counts, tissue area and density are reported per slide.
Needs the `cellcount` extra.

In [ ]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository whether Jupyter starts at its root or in how_to_use/."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "rocqipath").is_dir():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"          # your slides (kept out of git)
RESULTS_ROOT = PROJECT_ROOT / "results"    # outputs (kept out of git)
DEMO_ROOT = PROJECT_ROOT / "notebook_demo_outputs"  # synthetic examples

import rocqipath as rp

print(f"RocqiPath {rp.__version__}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")

In [ ]:
IHC_SLIDES = DATA_ROOT / "wsi" / "cd8"     # a folder, one slide, or rp.align output
OUTPUT_DIR = RESULTS_ROOT / "counts"

count_settings = dict(
    label="CD8",
    target_magnification=20.0,
    source_magnification=None,   # set only when metadata is absent
    patch_size=512,
    tissue_threshold=0.10,
    min_cell_area=50,            # pixels at target magnification
    max_cell_area=1000,
)

RUN_SYNTHETIC_DEMO = True
RUN_BATCH = False
RUN_COMPARISON = False

## Synthetic smoke test

48 brown "cells" on a plain TIFF — a software check, not a biological one.
Cell areas are in pixels² at the target magnification, so revalidate
`min_cell_area` and `max_cell_area` when you change magnification.

In [ ]:
from PIL import Image, ImageDraw

demo_slide = DEMO_ROOT / "cell_counting" / "synthetic_cd8.tif"
demo_slide.parent.mkdir(parents=True, exist_ok=True)
canvas = Image.new("RGB", (512, 512), (220, 190, 205))
draw = ImageDraw.Draw(canvas)
for row in range(6):
    for col in range(8):
        x, y = 45 + col * 58, 55 + row * 72
        draw.ellipse((x - 10, y - 10, x + 10, y + 10), fill=(180, 125, 70))
        draw.ellipse((x - 6, y - 6, x + 6, y + 6), fill=(85, 42, 18))
canvas.save(demo_slide)

if RUN_SYNTHETIC_DEMO:
    demo = rp.count_cells(demo_slide, DEMO_ROOT / "cell_counting" / "out",
                          **{**count_settings, "label": "synthetic", "source_magnification": 20.0})
    print(demo.summary["results"][0])

## Count a cohort

A folder is counted slide by slide (non-recursively), with one JSON per
slide and a batch summary. The density denominator is the per-pixel tissue
mask; verify scanner microns-per-pixel before interpreting cells/mm².

In [ ]:
if RUN_BATCH:
    counts = rp.count_cells(IHC_SLIDES, OUTPUT_DIR, **count_settings)
    for slide in counts.summary["results"]:
        print(f"{slide['slide']:40s} {slide['total_positive']:>8,} cells  {slide['density_per_mm2']:>10,.1f}/mm²")
else:
    print("Set RUN_BATCH=True after editing IHC_SLIDES.")

## Compare real and predicted IHC

Both slides are counted on the same grid (they must have the same size at the
target magnification, e.g. after alignment), with per-patch figures and an
Excel sheet.

In [ ]:
REAL_SLIDE = DATA_ROOT / "wsi" / "Sample_0001_cd8_gt.svs"
PREDICTED_SLIDE = DATA_ROOT / "wsi" / "Sample_0001_cd8_pred.tif"

if RUN_COMPARISON:
    comparison = rp.count_cells(REAL_SLIDE, OUTPUT_DIR, compare_to=PREDICTED_SLIDE,
                                **{**count_settings, "paired_source_magnification": 20.0},
                                max_plots=10, dpi=300)
    print(comparison.summary["result"])
else:
    print("Set RUN_COMPARISON=True after aligning both slides.")

**Validation checklist** — review masks on weak, strong, necrotic and folded
regions; confirm one component ≈ one cell; validate area thresholds at each
magnification; confirm microns-per-pixel; report the gate, thresholds,
magnification and cohort in your methods.